In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from db.connection import get_session
from evaluation.recall import evaluate_recall, load_ground_truth, dataset_stats
from evaluation.mlflow_log import log_eval_to_mlflow
from similarity.weights import SimilarityWeights

# --- Config: change these and re-run to compare ---
WEIGHTS = SimilarityWeights()  # uses defaults from config
TOP_K = 5

In [2]:
session = get_session()
pairs = load_ground_truth(session)
print(f"{len(pairs)} ground truth pairs loaded ({len(pairs) * 2} bidirectional evaluations)")

result = evaluate_recall(session, pairs, weights=WEIGHTS, top_k=TOP_K)
print(f"Evaluation complete in {result.duration_s}s")

2026-03-06 17:54:17,572 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-06 17:54:17,573 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-06 17:54:17,577 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-06 17:54:17,577 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-06 17:54:17,580 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-06 17:54:17,580 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-06 17:54:17,581 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-06 17:54:17,583 INFO sqlalchemy.engine.Engine SELECT ground_truth_pairs.id AS ground_truth_pairs_id, ground_truth_pairs.species_a AS ground_truth_pairs_species_a, ground_truth_pairs.species_b AS ground_truth_pairs_species_b, ground_truth_pairs.danger_note AS ground_truth_pairs_danger_note, ground_truth_pairs.source AS ground_truth_pairs_source, ground_truth_pairs.notes AS ground_truth_pairs_notes 
FROM ground_truth_pairs
2026-03-06 17:54:17,584 INFO sqlalchemy.engine.E

Error querying Cortinarius Seriocybe: Species not found: 'Cortinarius Seriocybe'


2026-03-06 17:54:59,460 INFO sqlalchemy.engine.Engine SELECT reconciled_species.id AS reconciled_species_id, reconciled_species.scientific_name AS reconciled_species_scientific_name, reconciled_species.common_names AS reconciled_species_common_names, reconciled_species.family AS reconciled_species_family, reconciled_species.genus AS reconciled_species_genus, reconciled_species.features_json AS reconciled_species_features_json, reconciled_species.edibility AS reconciled_species_edibility, reconciled_species.known_toxins AS reconciled_species_known_toxins, reconciled_species.known_lookalikes AS reconciled_species_known_lookalikes, reconciled_species.reconciliation_confidence AS reconciled_species_reconciliation_confidence, reconciled_species.needs_review AS reconciled_species_needs_review, reconciled_species.review_notes AS reconciled_species_review_notes, reconciled_species.human_overrides AS reconciled_species_human_overrides, reconciled_species.reconciled_at AS reconciled_species_reco

Error querying Cortinarius Seriocybe: Species not found: 'Cortinarius Seriocybe'


2026-03-06 17:55:00,316 INFO sqlalchemy.engine.Engine SELECT reconciled_species.id AS reconciled_species_id, reconciled_species.scientific_name AS reconciled_species_scientific_name, reconciled_species.common_names AS reconciled_species_common_names, reconciled_species.family AS reconciled_species_family, reconciled_species.genus AS reconciled_species_genus, reconciled_species.features_json AS reconciled_species_features_json, reconciled_species.edibility AS reconciled_species_edibility, reconciled_species.known_toxins AS reconciled_species_known_toxins, reconciled_species.known_lookalikes AS reconciled_species_known_lookalikes, reconciled_species.reconciliation_confidence AS reconciled_species_reconciliation_confidence, reconciled_species.needs_review AS reconciled_species_needs_review, reconciled_species.review_notes AS reconciled_species_review_notes, reconciled_species.human_overrides AS reconciled_species_human_overrides, reconciled_species.reconciled_at AS reconciled_species_reco

In [3]:
# Recall@K summary
for k in [1, 3, 5]:
    print(f"Recall@{k}: {result.hits_at(k)}/{result.total} = {result.recall_at(k):.1%}")

if result.errors:
    print(f"\nErrors (species not found / missing embeddings): {result.errors}")

Recall@1: 11/62 = 17.7%
Recall@3: 19/62 = 30.6%
Recall@5: 23/62 = 37.1%

Errors (species not found / missing embeddings): 2


In [4]:
# Log to MLflow
stats = dataset_stats(session)
ok = log_eval_to_mlflow(result, run_name=None, dataset_params=stats)
if ok:
    print("Logged to MLflow (experiment: eval_retrieval). View at http://localhost:5000")
else:
    print("MLflow logging skipped (service unavailable).")

2026-03-06 17:55:02,029 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM (SELECT reconciled_species.id AS reconciled_species_id, reconciled_species.scientific_name AS reconciled_species_scientific_name, reconciled_species.common_names AS reconciled_species_common_names, reconciled_species.family AS reconciled_species_family, reconciled_species.genus AS reconciled_species_genus, reconciled_species.features_json AS reconciled_species_features_json, reconciled_species.edibility AS reconciled_species_edibility, reconciled_species.known_toxins AS reconciled_species_known_toxins, reconciled_species.known_lookalikes AS reconciled_species_known_lookalikes, reconciled_species.reconciliation_confidence AS reconciled_species_reconciliation_confidence, reconciled_species.needs_review AS reconciled_species_needs_review, reconciled_species.review_notes AS reconciled_species_review_notes, reconciled_species.human_overrides AS reconciled_species_human_overrides, reconciled_species.reconc

/Users/aless/Desktop/Projects/mushroom-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run overjoyed-loon-567 at: http://localhost:5000/#/experiments/1/runs/a01aba82fac741228a1a52186682311f
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged to MLflow (experiment: eval_retrieval). View at http://localhost:5000


In [5]:
# Hits — successful retrievals sorted by rank
hits = [r for r in result.pair_results if r.rank is not None]
df_hits = pd.DataFrame([vars(r) for r in hits]).sort_values("rank")
sim_cols = ["query", "target", "rank", "sim_overall",
            "sim_macro_visual", "sim_structural", "sim_flesh_sensory",
            "sim_microscopic_lab", "sim_ecological", "sim_taxonomic", "sim_numeric"]
display(df_hits[[c for c in sim_cols if c in df_hits.columns]].round(3))

,query,target,rank,sim_overall,sim_macro_visual,sim_structural,sim_flesh_sensory,sim_microscopic_lab,sim_ecological,sim_taxonomic,sim_numeric
11,Amanita pantherina,Amanita rubescens,1,0.867,0.964,0.869,0.817,0.610,0.970,0.825,0.596
20,Hypholoma fasciculare,Hypholoma capnoides,1,0.879,0.910,0.916,0.784,0.605,0.947,0.826,0.800
4,Boletus edulis,Tylopilus felleus,1,0.856,0.933,0.915,0.886,0.781,0.924,0.854,0.511
5,Tylopilus felleus,Boletus edulis,1,0.856,0.933,0.915,0.886,0.781,0.924,0.854,0.511
16,Macrolepiota procera,Macrolepiota rhacodes,1,0.808,0.926,0.913,0.940,0.552,0.844,0.858,0.329
15,Coprinopsis atramentaria,Coprinus comatus,1,0.831,0.953,0.892,0.826,0.984,0.847,0.867,0.378
9,Galerina marginata,Kuehneromyces mutabilis,1,0.847,0.902,0.868,0.900,0.848,0.980,0.763,0.532
21,Hypholoma capnoides,Hypholoma fasciculare,1,0.879,0.910,0.916,0.784,0.605,0.947,0.826,0.800
12,Lactarius deliciosus,Lactarius torminosus,1,0.824,0.917,0.759,0.826,0.911,0.896,0.851,0.552
13,Lactarius torminosus,Lactarius deliciosus,1,0.824,0.917,0.759,0.826,0.911,0.896,0.851,0.552


In [6]:
# Misses — target not found in top-K
misses = [r for r in result.pair_results if r.rank is None]
df_misses = pd.DataFrame([vars(r) for r in misses])
display(df_misses[["query", "target", "error"]])

,query,target,error
0,Amanita caesarea,Amanita phalloides,None
1,Amanita phalloides,Amanita caesarea,None
2,Agaricus campestris,Amanita phalloides,None
3,Amanita phalloides,Agaricus campestris,None
4,Cantharellus cibarius,Omphalotus olearius,None
5,Omphalotus olearius,Cantharellus cibarius,None
6,Cantharellus cibarius,Hygrophoropsis aurantiaca,None
7,Hygrophoropsis aurantiaca,Cantharellus cibarius,None
8,Morchella esculenta,Gyromitra esculenta,None
9,Gyromitra esculenta,Morchella esculenta,None


In [ ]:
# Weight sweep: precompute once, rescore with 200 random configs
from evaluation.sweep import sweep_weights

sweep_results = sweep_weights(session, pairs, n_configs=200, seed=42)

df_sweep = pd.DataFrame([
    {
        "R@1": r.recall_at[1],
        "R@3": r.recall_at[3],
        "R@5": r.recall_at[5],
        **{f: r.weights[f] for f in r.weights if f != "body_form_filter"},
        "body_form_filter": r.weights["body_form_filter"],
    }
    for r in sweep_results
])
print(f"Swept {len(sweep_results)} configs. Top 20:")
display(df_sweep.head(20).round(4))